# Pràctica progressiva — Anàlisi de vendes d'una botiga

## Encàrrec professional
Treballes com a tècnic/a d'anàlisi de dades d'una botiga. La persona responsable comercial t'ha lliurat un CSV de vendes i necessita un resum fiable per prendre decisions. Hauràs de preparar les dades, justificar les regles aplicades i comunicar els resultats a un públic no tècnic.

## Objectius
- Llegir CSV amb `csv.DictReader` i explorar-ne l'estructura.
- Validar, normalitzar i convertir dades amb funcions pròpies.
- Separar dades vàlides, corregibles i rebutjades amb traçabilitat.
- Calcular indicadors i fer agrupacions.
- Verificar els resultats i generar fitxers d'informe.
- Formular conclusions prudents i defensar les decisions.

**Fitxers:** posa `vendes_botiga.csv` al mateix directori que aquest notebook.

La pràctica es divideix en fases guiades. Utilitza només la biblioteca estàndard de Python.

## Fase 1 — Inspecció guiada

Comença per entendre el fitxer abans de modificar-lo. Cada fila és una línia de venda; els camps són textuals quan es llegeixen del CSV.

In [2]:
import csv
from datetime import datetime

nom_fitxer = "vendes_botiga.csv"

# EXERCI 1.0: Obre el fitxer 'nom_fitxer' en mode lectura, llegeix-lo amb csv.DictReader
# i desa els camps a la variable 'camps' i els registres com una llista a 'registres'.

with open(nom_fitxer, mode="r", encoding="utf-8", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)

# Comprovacions
print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
if registres:
    print("Exemple de primer registre:", registres[0])

Camps del CSV: ['data', 'producte', 'categoria', 'unitats', 'preu_unitari', 'ciutat']
Nombre total de registres: 19
Exemple de primer registre: {'data': '2026-01-03', 'producte': 'Llibreta', 'categoria': 'Papereria', 'unitats': '12', 'preu_unitari': '2.50', 'ciutat': 'Barcelona'}


### Exercici 1.1
Calcula quants registres tenen almenys un camp buit o format només per espais. No modifiquis les dades encara.

**Pista:** Pots fer servir `.strip() == ""` sobre cada valor del diccionari per comprovar si un camp està buit o té només espais.

In [ ]:
# GUIA PAS A PAS:
# 1. Inicialitza el comptador 'registres_incomplets' a 0.
# 2. Revisa cada registre de la llista 'registres'.
# 3. Comprova si algun dels seus valors està buit després d'aplicar .strip().
# 4. Incrementa el comptador si es detecta algun camp buit.

#

registres_incomplets = 0

for reg in registres:
    # TODO: Comprova si hi ha algun camp buit en la fila reg
    # Pista: utilitza any(...) o un bucle sobre reg.values()

    if any(value.strip()== " " for value in reg.values()):
     registres_incomplets +=1
    pass

    #concept1: value.strip()=="" -> check wether after removing spaces if value is empty
    #concept2: values() -> in key-value pairs it gives you the values
    #concept3: any() -> returns true if at least one item satifies a condition

print(f"Registres amb almenys un camp buit/espais: {registres_incomplets}")

Registres amb almenys un camp buit/espais: 0


### Exercici 1.2

**Reflexiona:** quines dades són imprescindibles per calcular l'import d'una venda? Quines són necessàries per segmentar-la/classificar-la?

calculate sale : unitats and preu_unitari
identify what was sold : producte
classify sales : categoria, ciutat, data
produce a complete report : all relevant fields
import = unitats * preu unitari

## Fase 2 — Regles de qualitat i neteja guiada

Cal aplicar als registres de vendes aquestes regles:
1. Eliminar espais sobrants dels camps textuals (`strip()`).
2. `unitats` ha de ser un enter estrictament positiu (> 0).
3. `preu_unitari` ha de ser un nombre positiu (> 0); s'accepten coma o punt decimal.
4. La data ha de tenir el format `AAAA-MM-DD` i ser una data vàlida.
5. Producte, categoria i ciutat són camps descriptius obligatoris.
6. Un registre amb valors numèrics o data no interpretables no entra als càlculs. No inventis valors per completar-los (imputació).

### Exercici 2 — Funcions de conversió amb gestió d'errors
Implementa funcions de conversió robustes utilitzant blocs `try...except`.

In [ ]:
def converteix_enter_positiu(valor: str):
    """Retorna un int positiu (>0) o None si no és vàlid."""
    #def : defines a function
    #(valor: str): documents the expected input
    try:
        v = int(str(valor).strip()) #str(valor):It converts the input to text before stripping whitespace and converting it to an integer
        return v if v > 0 else None
    except (ValueError, TypeError):
         return None
    #concept4: try and except
    #concept5: None-> returns one if if condition not applied,
    #it can mean if condition is failed like: if val_convertit is None:

def converteix_preu(valor: str):
    """Accepta coma o punt decimal; retorna float positiu (>0) o None si no és vàlid."""
    # TODO: Neteja espais, reemplaça ',' per '.', converteix a float i comprova que sigui > 0
    try:
         v = float(str(valor).strip().replace(",",".")) # python can convert 5.2 to num but 5,2 no
         return v if v > 0 else None
    except (ValueError, TypeError):
             return None
    #concept6: value.replace("what","with")

def data_valida(valor: str) -> bool:
    """Retorna True si el text correspon a una data vàlida en format AAAA-MM-DD."""
    # TODO: Utilitza datetime.strptime(..., "%Y-%m-%d") dins d'un try-except
    try:
        datetime.strptime(valor,"%Y-%m-%d")
        return True

    except(ValueError,TypeError):
        return False
   #concept7: datetime.strptime(valor,"%Y-%m-%d"): convert valor input into a date in format yyyy-mm-dd, and python checks wether the date actually exists like feb 30 returns false

### Exercici 3 — Neteja amb traçabilitat

Classifica les dades en tres estructures:
- `vendes_netes`: llista de diccionaris amb dades convertides i camp `import` inclòs.
- `rebutjades`: llista de registres no vàlids.
- `incidencies`: llista de registres/incidències amb el detall per a la traçabilitat (`fila_csv`, `tipus`, `camp`, `valor_original`, `accio`).

In [ ]:
vendes_netes = []
incidencies = []# a record of problems and corrections
rebutjades = []

obligatoris = ["producte", "categoria", "ciutat"] # fields that must not be empty

for idx, reg in enumerate(registres, start=2): #concep8: enumerate()->loop through a sequence while also getting a counter,Why does your code use start=2? The first row of a CSV is the header, so the first data record is on line 2.
    es_valid = True #assumes the record is valid until a problrm is found
    reg_net = {} #creates an empty dictionary to build the clean sale
    
    # TODO PAS 1: Comprovació de camps obligatoris
    # - Comprova si producte, categoria i ciutat estan buits.
    # - Si estan buits: marca es_valid = False i afegeix una incidència 'REBUTJAT'.
    # - Si tenen espais de més: afegeix una incidència 'CORRECCIO' i guarda el text net a reg_net.
    for camp in obligatoris:
            val_original = reg.get(camp, "")#If the key is missing, it returns "" rather than raising a KeyError.
            val_net = str(val_original).strip()# checks wether the cleaned string is empty
            if not val_net: #checks wether the cleaned string is empty
                es_valid = False
                incidencies.append({#adds items to the list incidencies
                    "fila_csv": idx, "tipus": "REBUTJAT", "camp": camp,
                    "valor_original": val_original, "accio": "Camp obligatori buit"
                })
            else:
                if val_original != val_net: #If the cleaned text is empty
                    incidencies.append({
                        "fila_csv": idx, "tipus": "CORRECCIO", "camp": camp,
                        "valor_original": val_original, "accio": "Espais sobrants eliminats"
                    })
                reg_net[camp] = val_net #stores the clean value under the appropriate key
    
    # TODO PAS 2: Conversió d'unitats
    # - Utilitza converteix_enter_positiu(reg.get("unitats"))

    val_original = reg.get("unitats","")
    val_convertit = converteix_enter_positiu(val_original)
    # - Si és None: es_valid = False i afegeix incidència 'REBUTJAT'.
    if val_convertit is None:
        es_valid = False
        incidencies.append({
        "fila_csv": idx,
        "tipus": "REBUTJAT",
        "camp": "unitats",
        "valor_original": val_original,
        "accio": "Unitats no vàlides"
    })

    # - Si és vàlid: desa el valor convertir a reg_net["unitats"].
    else:
         reg_net["unitats"]= val_convertit


    # TODO PAS 3: Conversió de preu unitari
    # - Utilitza converteix_preu(reg.get("preu_unitari"))
    preu_original= reg.get("preu_unitari", "")
    preu_convertit= converteix_preu(preu_original)
    # - Detecta si tenia coma per afegir una incidència 'CORRECCIO'.
    if preu_convertit is None:
         es_valid = False
         incidencies.append({
        "fila_csv": idx,
        "tipus": "REBUTJAT",
        "camp": "preu_unitari",
        "valor_original": preu_original,
        "accio": "Preu no vàlid"
    })
else:
    if "," in str(preu_original):
        incidencies.append({
              "fila_csv": idx,
            "tipus": "CORRECCIO",
            "camp": "preu_unitari",
            "valor_original": preu_original,
            "accio": "Coma decimal convertida a punt"
        })

    reg_net["preu_unitari"] = preu_convertit

    # - Si és None: es_valid = False i afegeix incidència 'REBUTJAT'.
    
    # TODO PAS 4: Validació de data
    # - Utilitza data_valida(reg.get("data"))
    data_original = reg.get("data", "")

if not data_valida(data_original):
    es_valid = False
    incidencies.append({
        "fila_csv": idx,
        "tipus": "REBUTJAT",
        "camp": "data",
        "valor_original": data_original,
        "accio": "Data no vàlida"
    })
else:
    reg_net["data"] = data_original

    # - Si no és vàlida: es_valid = False i afegeix incidència 'REBUTJAT'.
    
    # TODO PAS 5: Classificació final
    # - Si es_valid és True: calcula reg_net["import"] = round(unitats * preu, 2)
    #   i afegeix reg_net a vendes_netes.
if es_valid:
        reg_net["import"] = round(
            reg_net["unitats"] * reg_net["preu_unitari"], 2
        )
        vendes_netes.append(reg_net)
else:
        rebutjades.append(reg)
    # - Si es_valid és False: afegeix reg a rebutjades.

print(f"Llegits: {len(registres)}")
print(f"Vàlids: {len(vendes_netes)}")
print(f"Rebutjats: {len(rebutjades)}")
print(f"Incidències documentades: {len(incidencies)}")

## Fase 3 — Indicadors de negoci

Calcula els indicadors globals i realitza les agrupacions per categoria i ciutat.

In [ ]:
# GUIA DE CÀLCUL:
# 1. Suma total d'unitats i d'imports a partir de 'vendes_netes'.
# 2. Càlcul de mitjanes: preu unitari mitjà vs import mitjà per línia.
# 3. Agrupació en diccionaris acumulant la facturació per clau.

# TODO: Calcula els indicadors totals
total_unitats = sum(v["unitats"] for v in vendes_netes)
facturacio_total = sum(v["import"] for v in vendes_netes)
preu_mitja =  (
    sum(v["preu_unitari"] for v in vendes_netes)
    / len(vendes_netes)
    if vendes_netes else 0
)

import_mitja = (
    sum(v["import"] for v in vendes_netes)
    / len(vendes_netes)
    if vendes_netes else 0
)

# TODO: Agrupa la facturació per categoria i ciutat utilitzant diccionaris
resum_categoria = {}
resum_ciutat = {}

for v in vendes_netes:
    cat = v["categoria"]
    ciu = v["ciutat"]
    resum_categoria[cat] = round(resum_categoria.get(cat, 0.0) + v["import"], 2)
    resum_ciutat[ciu] = round(
        resum_ciutat.get(ciu, 0.0) + v["import"], 2
    )

print(f"Total unitats: {total_unitats}")
print(f"Facturació total: {facturacio_total} €")
print(f"Preu unitari mitjà: {preu_mitja} €")
print(f"Import mitjà per línia: {import_mitja} €")
print("Facturació per categoria:", resum_categoria)
print("Facturació per ciutat:", resum_ciutat)

### Control de qualitat
Validació automàtica de la consistència de les dades mitjançant assercions (`assert`). Omple la informació corresponent per verificar les teves dades.

In [ ]:
# TODO 1: Comprova que la suma de vàlids i rebutjats sigui igual al total llegit
assert len(vendes_netes) + len(rebutjades) == len(registres), "Error: La suma de vàlids i rebutjats no quadrat amb el total."

# TODO 2: Comprova que tots els valors de unitats, preu_unitari i import siguin positius (>0)
# assert all(...), "Error: Hi ha valors nuls o negatius"
assert all(
    v["unitats"] > 0
    and v["preu_unitari"] > 0
    and v["import"] > 0
    for v in vendes_netes
), "There are zero or negative values."

assert abs(
    sum(resum_categoria.values()) - facturacio_total
) < 0.01, "Category totals do not match total revenue."

# TODO 3: Comprova que la suma del diccionari resum_categoria quadri amb facturacio_total
# assert abs(sum(resum_categoria.values()) - facturacio_total) < 0.01, "Error en la suma de categories"
top_categoria = (
    max(resum_categoria.items(), key=lambda x: x[1])
    if resum_categoria else (None, 0)
)
top_ciutat = (
    max(resum_ciutat.items(), key=lambda x: x[1])
    if resum_ciutat else (None, 0)
)
print("Executa aquesta cel·la quan hagis completat les assercions!")

## Fase 4 — Interpretació i comunicació

Respon a les preguntes analítiques identificant els màxims i avaluant les conclusions de negoci.

In [ ]:
# TODO 1: Troba la categoria i ciutat amb més facturació
# Pista: pots fer servir max(resum_categoria.items(), key=lambda x: x[1])
top_categoria = max(resum_categoria.items(), key=lambda x: x[1]) if resum_categoria else (None, 0)
top_ciutat = (
    max(resum_ciutat.items(), key=lambda x: x[1])
    if resum_ciutat else (None, 0)
)

# TODO 2: Troba el producte amb més unitats venudes
# Pista: necessitaràs agrupar primer les unitats totals venudes per producte
unitats_per_producte = {}
for v in vendes_netes:
    producte = v["producte"]
    unitats_per_producte[producte] = (
        unitats_per_producte.get(producte, 0) + v["unitats"]
    )
# recorre vendes_netes per anar sumant les unitats venudes de cada producte, ves guardant els totals per producte al diccionari unitats_per_producte
top_producte = (
    max(unitats_per_producte.items(), key=lambda x: x[1])
    if unitats_per_producte else (None, 0)
)

print(f"1. Categoria amb més facturació: {top_categoria[0]} ({top_categoria[1]} €)")
print(f"   Ciutat amb més facturació: {top_ciutat[0]} ({top_ciutat[1]} €)")
print(f"2. Producte amb més unitats venudes: {top_producte[0]} ({top_producte[1]} unitats)")

## Fase 5 — Lliurament professional

Generació dels tres fitxers de sortida: `informe_vendes.txt`, `incidencies.csv` i `vendes_netes.csv`.

In [ ]:
nom_informe = "informe_vendes.txt"
nom_incidencies = "incidencies.csv"
nom_netes = "vendes_netes.csv"

# TODO 1: Genera 'vendes_netes.csv' utilitzant csv.DictWriter amb les dades de vendes_netes
if vendes_netes:
    with open(nom_netes, mode="w", encoding="utf-8", newline="") as f:
        escriptor = csv.DictWriter(f, fieldnames=vendes_netes[0].keys())
        escriptor.writeheader()
        escriptor.writerows(vendes_netes)

# TODO 2: Genera 'incidencies.csv' utilitzant csv.DictWriter amb els camps:
# ["fila_csv", "tipus", "camp", "valor_original", "accio"]
camps_incidencies = [
    "fila_csv", "tipus", "camp", "valor_original", "accio"
]

with open(nom_incidencies, mode="w", encoding="utf-8", newline="") as f:
    escriptor = csv.DictWriter(f, fieldnames=camps_incidencies)
    escriptor.writeheader()
    escriptor.writerows(incidencies)

# TODO 3: Genera l'informe de text 'informe_vendes.txt' amb un resum executiu que inclogui:
# - Data de generació
# - Cobertura de dades (vàlids vs rebutjats)
# - Indicadors globals (unitats, facturació, mitjanes)
# - Desglossament per categoria i ciutat
with open(nom_informe, mode="w", encoding="utf-8") as f:
    f.write("SALES REPORT\n")
    f.write(f"Generated: {datetime.now()}\n")
    f.write(f"Records read: {len(registres)}\n")
    f.write(f"Valid records: {len(vendes_netes)}\n")
    f.write(f"Rejected records: {len(rebutjades)}\n")
    f.write(f"Total units: {total_unitats}\n")
    f.write(f"Total revenue: {facturacio_total:.2f} EUR\n")
    f.write(f"Average unit price: {preu_mitja:.2f} EUR\n")
    f.write(f"Average amount per sale: {import_mitja:.2f} EUR\n")
    f.write(f"Revenue by category: {resum_categoria}\n")
    f.write(f"Revenue by city: {resum_ciutat}\n")

print("Fitxers d'informe i exportació generats correctament.")